# NLP Multi-Label Emotion Logistic Regression Classifier Test Run

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

In [3]:
# Load the training data.
train_df = pd.read_csv(
    "../data/preprocessed/train.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

train_df.head()

,text,emotion_ids,id
0,My favourite food is anything I didn't have to...,27,eebbqej
1,"Now if he does off himself, everyone will thin...",27,ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj
3,To make her feel threatened,14,ed7ypvh
4,Dirty Southern Wankers,3,ed0bdzj


In [5]:
print(train_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [4]:
print(train_df.shape)

(43410, 3)


In [7]:
# Load the official dev split (used as validation for now)
val_df = pd.read_csv(
    "../data/preprocessed/dev.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

In [8]:
val_df.head()

,text,emotion_ids,id
0,Is this in New Orleans?? I really feel like th...,27,edgurhb
1,"You know the answer man, you are programmed to...","4,27",ee84bjg
2,I've never been this sad in my life!,25,edcu99z
3,The economy is heavily controlled and subsidiz...,"4,27",edc32e2
4,He could have easily taken a real camera from ...,20,eepig6r


In [13]:
print(val_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [14]:
print(val_df.shape)

(5426, 3)


In [16]:
# Load in test.
test_df = pd.read_csv(
    "../data/preprocessed/test.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

In [17]:
test_df.head()

,text,emotion_ids,id
0,I’m really sorry about your situation :( Altho...,25,eecwqtt
1,It's wonderful because it's awful. At not with.,0,ed5f85d
2,"Kings fan here, good luck to you guys! Will be...",13,een27c3
3,"I didn't know that, thank you for teaching me ...",15,eelgwd1
4,They got bored from haunting earth for thousan...,27,eem5uti


In [18]:
print(test_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [19]:
print(test_df.shape)

(5427, 3)


In [20]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43410 entries, 0 to 43409
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   text         43410 non-null  str  
 1   emotion_ids  43410 non-null  str  
 2   id           43410 non-null  str  
dtypes: str(3)
memory usage: 4.2 MB


In [21]:
train_df['emotion_ids'].value_counts()

emotion_ids
27            12823
0              2710
4              1873
15             1857
1              1652
              ...  
0,12,13,26        1
1,2,5,17          1
13,14             1
3,9,12            1
0,1,18            1
Name: count, Length: 711, dtype: int64

In [22]:
train_df['emotion_ids'].value_counts().head(20)

emotion_ids
27    12823
0      2710
4      1873
15     1857
1      1652
3      1451
18     1427
10     1402
7      1389
2      1025
20      861
6       858
17      853
25      817
26      720
9       709
5       649
22      586
13      510
11      498
Name: count, dtype: int64

In [24]:
print(train_df['emotion_ids'].nunique())

711


In [25]:
train_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    36308
True      7102
Name: count, dtype: int64

In [26]:
total_count = len(train_df)
multi_labeled_count = train_df['emotion_ids'].str.contains(',').sum()
single_labeled_count = (~train_df['emotion_ids'].str.contains(',')).sum()

print(f"Total data points: {total_count}")
print(f"Single-labeled data points: {single_labeled_count}")
print(f"Multi-labeled data points: {multi_labeled_count}")

Total data points: 43410
Single-labeled data points: 36308
Multi-labeled data points: 7102


In [27]:
val_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    4548
True      878
Name: count, dtype: int64

In [28]:
test_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    4590
True      837
Name: count, dtype: int64

* Look into a multi-hot label matrix.
  - Binary vector of 1s and 0s.

  - A FIXED-size numeric-tensor.

  - UNLIKE one-hot encoding, MULTIPLE positions can be 1s at once - hence "multi-hot"

  - EX: (admiration, amusement, love) --> (0, 1, 18) --> (1, 1, 0, ...., 1, ...) where the last '1' is at position 18.

In [29]:
from sklearn.preprocessing import MultiLabelBinarizer

In [30]:
emotion_names = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

In [31]:
train_df["emotion_id_list"] = (
  train_df['emotion_ids'].str.split(',').apply(lambda ids: [int(i) for i in ids])
)

In [32]:
train_df.tail()

,text,emotion_ids,id,emotion_id_list
43405,Added you mate well I’ve just got the bow and ...,18,edsb738,[18]
43406,Always thought that was funny but is it a refe...,6,ee7fdou,[6]
43407,What are you talking about? Anything bad that ...,3,efgbhks,[3]
43408,"More like a baptism, with sexy results!",13,ed1naf8,[13]
43409,Enjoy the ride!,17,eecwmbq,[17]


In [33]:
mlb = MultiLabelBinarizer(classes=range(len(emotion_names)))

y = mlb.fit_transform(train_df['emotion_id_list'])

print(y.shape)

(43410, 28)


In [35]:
train_df[train_df['emotion_ids'].str.contains(',')]

,text,emotion_ids,id,emotion_id_list
7,We need more boards and to create a bit more s...,"8,20",ef4qmod,"[8, 20]"
11,"Aww... she'll probably come around eventually,...","1,4",edex4ki,"[1, 4]"
15,"Shit, I guess I accidentally bought a Pay-Per-...","3,12",edivtm3,"[3, 12]"
19,Maybe that’s what happened to the great white ...,"6,22",eczq8zg,"[6, 22]"
20,"I never thought it was at the same moment, but...","6,9,27",efdlhs1,"[6, 9, 27]"
...,...,...,...,...
43382,"goat handshake denied Personally, I just thin...","14,27",ef2m53x,"[14, 27]"
43383,it's horrid :/,"14,27",edlbr3j,"[14, 27]"
43388,Fuck these trendy hipster joints. Give me my s...,"2,3",ee3nyiy,"[2, 3]"
43395,Sorry I kind of took it like you were flexing ...,"1,24",eelhhzc,"[1, 24]"


In [36]:
y[:1]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1]])

In [37]:
# Look at the matrix for MULTI-LABELS
y[43382: 43384]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1]])